# Policy Drift OpenEnv — Training Notebook

End-to-end **SFT warm-up + GRPO** fine-tune on the Policy Drift environment.

**Pipeline:**
1. Clone the repo and install deps (Unsloth + TRL + vLLM)
2. Generate training dataset from the env
3. Load Qwen 2.5 with LoRA adapters
4. Run SFT warm-up (1 epoch on correct-action pairs)
5. Run GRPO with 3 independent reward components
6. Offline eval: before / after-SFT / after-GRPO comparison
7. Save LoRA adapters (never naive-merge from 4-bit)

Set `QUICK_MODE = True` for a 5-minute pipeline validation (tiny model, small run).  
Set `QUICK_MODE = False` for the onsite run with HF compute credits.

## 1. Install dependencies

In [ ]:
%%capture
!pip install -q unsloth==2025.10.1 trl==0.12.1 datasets==3.0.0 transformers==4.46.0 accelerate==1.1.1
!pip install -q python-dotenv wandb
import os; os.environ['UNSLOTH_RETURN_LOGITS'] = '1'

## 2. Clone the repo (skip if running locally)

In [ ]:
import os
if not os.path.exists('OpenEnv'):
    !git clone https://github.com/shreyas-garg/OpenEnv.git
%cd OpenEnv
import sys; sys.path.insert(0, os.getcwd())

## 3. Pick run mode

`QUICK_MODE=True` → Qwen 2.5 0.5B, 50 episodes, 50 GRPO steps. ~5 min on Colab T4.  
`QUICK_MODE=False` → Qwen 2.5 3B, 800 episodes, 600 GRPO steps. Onsite only.

In [ ]:
os.environ['QUICK_MODE'] = 'true'       # <-- flip to 'false' onsite
os.environ['USE_WANDB'] = 'false'       # <-- flip to 'true' when you have a WANDB_API_KEY

## 4. Sanity check — env + dataset work

In [ ]:
from drift_env.dataset import build_dataset, dataset_stats
rows = build_dataset(n_episodes=10, start_seed=0)
print(dataset_stats(rows))
print('\nSample prompt (first 400 chars):\n' + rows[5]['prompt'][:400])

## 5. Run the full pipeline

In [ ]:
!python train.py

## 6. Inspect adapters + sample a trained rollout

After the run, LoRA adapters live in `./outputs/lora_adapters/`. Reload them in a fresh model to keep memory manageable if inspecting multiple checkpoints.

In [ ]:
!ls -la outputs/lora_adapters/ 2>/dev/null || echo 'train.py did not finish'

## Notes on scale-up for onsite

- Set `QUICK_MODE=false`, bump `MODEL_NAME` to `unsloth/Qwen2.5-3B-Instruct` (or `7B`).
- Use H100 or A100 — T4 can't do 3B with reasonable throughput.
- Enable wandb: `os.environ['WANDB_API_KEY']='...'`, `USE_WANDB='true'`.
- Save checkpoints every 100 steps once the curve is moving, in case you need to roll back.